In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import random
import os
from tqdm import tqdm # Biblioteca para barra de progresso

In [2]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """Função auxiliar para plotar um segmento de batimento de um DataFrame."""
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [3]:

def save_augmented(segment_df, output_dir, file_name):
    """Salva segmento em CSV, construindo o caminho completo."""
    # Garante que o DataFrame tenha as colunas no formato original para salvar
    augmented_df = segment_df.copy()
    
    if "amplitude" in augmented_df.columns:
        augmented_df["channel_0"] = augmented_df["amplitude"]
    
    # Seleciona e ordena as colunas
    if "type" not in augmented_df.columns: augmented_df['type'] = 'unknown' # Garante que a coluna 'type' exista
    if "sample #" not in augmented_df.columns: augmented_df['sample #'] = np.arange(len(augmented_df)) # Garante que a coluna 'sample #' exista

    augmented_df = augmented_df[["channel_0", "sample #", "type"]]
    
    # Constrói o caminho completo do arquivo
    full_path = os.path.join(output_dir, file_name)
    augmented_df.to_csv(full_path, index=False)

In [4]:

def augment_jitter(segment_df, save_dir, file_name, sigma_factor=0.02, variable_sigma=True, seed=None):
    rng = np.random.default_rng(seed)
    augmented_df = segment_df.copy()
    base_sigma = sigma_factor * np.std(segment_df["amplitude"])
    
    if variable_sigma:
        sigma_series = base_sigma * rng.lognormal(mean=0, sigma=0.25, size=len(segment_df))
    else:
        sigma_series = np.full(len(segment_df), base_sigma)
        
    noise = rng.normal(0, sigma_series)
    augmented_df["amplitude"] += noise
    
    # Salva o resultado chamando a função auxiliar
    save_augmented(augmented_df, save_dir, file_name)
    
    return augmented_df

In [5]:
# Coloque esta função no lugar da sua versão antiga
def get_beat_interval_robust_optimized(df, all_peaks, target_rows, nth=0, channel="channel_0"):
    """
    Versão otimizada que recebe um DataFrame e picos pré-calculados.
    """
    # Não precisa mais ler o CSV nem encontrar os picos aqui
    if nth >= len(target_rows):
        print(f"Aviso: nth={nth} está fora do alcance. Anotações encontradas: {len(target_rows)}")
        return None

    center_sample = int(target_rows.iloc[nth]["sample #"])

    # Encontra o índice do pico mais próximo da anotação
    center_peak_index = np.argmin(np.abs(all_peaks - center_sample))
    
    # Verificação de borda
    if center_peak_index == 0 or center_peak_index >= len(all_peaks) - 1:
        return None

    # Pega os picos vizinhos
    start_peak = all_peaks[center_peak_index - 1]
    end_peak = all_peaks[center_peak_index + 1]
    
    # Extrai o segmento
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()

    # Renomeia a coluna para o padrão "amplitude"
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    return segment_df

In [6]:
# --- Execução Otimizada para Múltiplos Arquivos ---
if __name__ == '__main__':
    csv_file = "mitbih_all_records_renumerada.csv"
    output_dir = "data_aug/"
    target_type = "R"
    num_augmentations = 7200
    
    # Garante que o diretório de saída exista
    os.makedirs(output_dir, exist_ok=True)
    
    # =================================================================
    # PASSO 1: LER E PRÉ-CALCULAR TUDO FORA DO LOOP
    # =================================================================
    print("1/3 - Carregando o arquivo CSV principal (apenas uma vez)...")
    df_main = pd.read_csv(csv_file)
    
    print("2/3 - Encontrando todas as anotações do tipo '{}'...".format(target_type))
    target_rows = df_main[df_main["type"] == target_type].reset_index(drop=True)
    
    if len(target_rows) < num_augmentations:
        print(f"Aviso: Pedido de {num_augmentations} aumentos, mas apenas {len(target_rows)} batimentos do tipo '{target_type}' foram encontrados.")
        num_augmentations = len(target_rows)

    print("3/3 - Detectando todos os picos R principais (apenas uma vez)...")
    peaks, _ = find_peaks(df_main["channel_0"].values, distance=180, height=0.25)
    
    # =================================================================
    # PASSO 2: EXECUTAR O LOOP RÁPIDO, SALVANDO A CADA ITERAÇÃO
    # =================================================================
    print(f"Gerando e salvando {num_augmentations} arquivos aumentados...")
    
    for i in tqdm(range(num_augmentations), desc="Gerando Arquivos"):
        # Pega o segmento rapidamente
        segment_df = get_beat_interval_robust_optimized(df_main, peaks, target_rows, nth=i)
        
        if segment_df is not None:
            # Define um nome de arquivo único para esta iteração
            file_name = f"{target_type}_beat_{i}_aug_jitter.csv"
            
            # Aplica o Jitter e salva o arquivo
            augment_jitter(
                segment_df=segment_df,
                save_dir=output_dir,
                file_name=file_name
            )
            
    print(f"Processo concluído! {num_augmentations} arquivos foram salvos em '{output_dir}'.")

1/3 - Carregando o arquivo CSV principal (apenas uma vez)...
2/3 - Encontrando todas as anotações do tipo 'R'...
3/3 - Detectando todos os picos R principais (apenas uma vez)...
Gerando e salvando 7200 arquivos aumentados...


Gerando Arquivos: 100%|█████████████████████████████████████████████████████████████| 7200/7200 [08:51<00:00, 13.54it/s]

Processo concluído! 7200 arquivos foram salvos em 'data_aug/'.
